In [41]:
import os
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable
from dotenv import load_dotenv 
from utilities.cypher_query_loader import CypherQueryLoader
from utilities.fbref import FBRefDataLoader, FBRefDataLoader
import json

In [42]:
# dylan instance details 

load_dotenv(dotenv_path = os.path.join(os.getcwd(),'config/.env'))

NEO4J_URI=os.getenv("NEO4J_URI")
NEO4J_USERNAME=os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD=os.getenv("NEO4J_PASSWORD")
CQL = CypherQueryLoader(os.path.join(os.getcwd(), 'queries/'))
DataLoader = FBRefDataLoader(os.path.join(os.getcwd(),'data/'))
DataFormatter = FBRefDataFormatter()

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

NameError: name 'FBRefDataFormatter' is not defined

In [27]:
data_directory = os.path.join(os.getcwd(),'data')
for season_directory in DataLoader.listdir(data_directory):
    print(season_directory)
    
    match_reports = DataLoader.load_data("fixtures_trial", season_directory)
    team_details = DataLoader.load_data("teams", season_directory)
    player_details = DataLoader.load_data("players", season_directory)

    player_details = add_team_name_to_player(player_details, team_details)
    game_weeks_data = [{"gw_number": f"GW{gw}", "season_name": season_directory} for gw in range(1, 39)]
    match_data, player_stats_data, goalkeeper_stats_data = DataFormatter.split_match_data(match_reports)

    gameweeks_season_query = CQL.load_query("season_gameweeks_creation")
    player_creation_query = CQL.load_query("player_creation")
    match_creation_query = CQL.load_query("match_creation")
    player_gamestats_query = CQL.load_query("player_game_stats")
    goalie_gamestats_query = CQL.load_query("goalie_game_stats")

    
    # Execute Queries
    with driver.session(database="neo4j") as session:
        session.run(gameweeks_season_query, parameters={"game_weeks": game_weeks_data})
        session.run(player_creation_query, parameters={"players": player_details, "season_name": season_directory})
        session.run(match_creation_query, parameters={"matches": match_data, "season_name": season_directory})
        session.run(player_gamestats_query, parameters={
            "player_stats": player_stats_data,
            "season_name": season_directory
        })
        session.run(goalie_gamestats_query, parameters={
            "goalkeeper_stats": goalkeeper_stats_data,
            "season_name": season_directory
        })

2021-2022
2022-2023
2023-2024
2024-2025


[#EFA5]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('5fe6b298.databases.neo4j.io', 7687)) (ResolvedIPv4Address(('35.205.213.74', 7687))): OSError('No data')


SessionExpired: Failed to read from defunct connection IPv4Address(('5fe6b298.databases.neo4j.io', 7687)) (ResolvedIPv4Address(('35.205.213.74', 7687)))